# Encode ESD (English) on Colab — SPARC features

Encodes the English subset of ESD (speakers 0011–0020, 5 emotions × 350 utterances each = 17,500 files) into articulatory features for use in our phoneme-indexed emotion delta table.

Hardened against Drive flakiness / session disconnects — resumes cleanly from what is already on Drive.

**Expected runtime:** ~1.5-2.5 hours on a T4 (≈2-3 credits). Plan: 1 credit buys ~1 hour of T4; with 3 credits you should finish in one session comfortably.

**Runtime setup:** Runtime → Change runtime type → T4 GPU.

In [ ]:
# Step 0: GPU check + Drive force-remount + shell-based existing-file listing
import os, subprocess, time
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"}')

from google.colab import drive
try:
    drive.flush_and_unmount()
except Exception:
    pass
drive.mount('/content/drive', force_remount=True)

DRIVE_FEATURES = '/content/drive/MyDrive/articulatory_tts/esd_features'
os.makedirs(DRIVE_FEATURES, exist_ok=True)


def shell_ls(path, timeout=600):
    r = subprocess.run(f"ls -1 '{path}'", shell=True,
                       capture_output=True, text=True, timeout=timeout)
    if r.returncode != 0:
        raise RuntimeError(f'ls failed (rc={r.returncode}): {r.stderr.strip()[:300]}')
    return [line.strip() for line in r.stdout.splitlines() if line.strip()]


def cleanup_stale_tmps(path):
    r = subprocess.run(f"find '{path}' -maxdepth 1 -name '*.tmp' -delete -print",
                       shell=True, capture_output=True, text=True, timeout=300)
    removed = [l for l in r.stdout.splitlines() if l.strip()]
    if removed:
        print(f'Cleaned up {len(removed)} stale .tmp files from prior crashes')
    return len(removed)


def build_done_stems(names):
    done = set()
    for n in names:
        if n.endswith('.npz') and n != 'norm_stats.npz':
            done.add(n[:-4])
    return done


print('Cleaning up any stale .tmp files on Drive...')
cleanup_stale_tmps(DRIVE_FEATURES)

print('\nListing existing Drive files via shell ls...')
t0 = time.time()
names = shell_ls(DRIVE_FEATURES)
done_stems = build_done_stems(names)
print(f'Already encoded on Drive: {len(done_stems)} files (listing took {time.time()-t0:.1f}s)')

In [ ]:
# Step 1: Install SPARC + deps
!pip install -q speech-articulatory-coding soundfile librosa

In [ ]:
# Step 2: Fetch ESD from HuggingFace mirror (no manual URL paste needed)
#
# Using sonchuate/ESD_dataset — a public HF mirror of the original ESD corpus
# (2.45 GB zip). Direct HF resolve URL is stable and resumable.
# Original source: https://hltsingapore.github.io/ESD/ (CC-BY-NC, research use).
#
# Structure inside the zip (VERIFIED locally before Colab run):
#   Emotion Speech Dataset/
#     0001/ .. 0010/  (Mandarin speakers — skip)
#     0011/ .. 0020/  (English speakers)
#       0011.txt                      <- tab-separated utt_id\ttext\temotion
#       Angry/   0011_000351.wav ...  <- 350 files
#       Happy/   0011_000701.wav ...  <- 350 files
#       Neutral/ 0011_000001.wav ...  <- 350 files
#       Sad/     0011_001051.wav ...  <- 350 files
#       Surprise/ 0011_001401.wav ... <- 350 files

import os, subprocess, glob

ESD_DIR = '/content/ESD'
ESD_ZIP = '/content/ESD.zip'
ESD_URL = 'https://huggingface.co/datasets/sonchuate/ESD_dataset/resolve/main/Emotional%20Speech%20Dataset%20(ESD).zip'
EXPECTED_TOP_DIR = 'Emotion Speech Dataset'  # exact folder name inside the zip

if os.path.exists(ESD_DIR) and os.listdir(ESD_DIR):
    print(f'ESD already extracted at {ESD_DIR}')
else:
    if not os.path.exists(ESD_ZIP):
        print('Downloading ESD (2.45 GB from HF) — resumable...')
        !wget -c --tries=0 --timeout=30 --read-timeout=60 "$ESD_URL" -O "$ESD_ZIP"
    else:
        print(f'Zip already downloaded at {ESD_ZIP}')

    print('Extracting (excluding __MACOSX metadata)...')
    # -q = quiet; exclude __MACOSX and .DS_Store junk
    !unzip -q "$ESD_ZIP" -x "__MACOSX/*" "*/.DS_Store" -d /content/

    src = f'/content/{EXPECTED_TOP_DIR}'
    if os.path.isdir(src):
        os.rename(src, ESD_DIR)
    elif not os.path.isdir(ESD_DIR):
        # Fallback: locate the folder containing numbered speaker subdirs
        found = None
        for root, dirs, _ in os.walk('/content', topdown=True):
            dirs[:] = [d for d in dirs if not d.startswith('.') and d not in ('drive', 'sample_data', '__MACOSX')]
            if sum(1 for d in dirs if d.isdigit() and len(d) == 4) >= 5:
                found = root
                break
        if found:
            os.rename(found, ESD_DIR)
        else:
            raise RuntimeError('Could not find ESD speaker folders after extract')

    !rm -f "$ESD_ZIP"
    print(f'Extracted to {ESD_DIR}')


# English speakers are IDs 0011-0020 in ESD
ENGLISH_SPEAKERS = [f'{i:04d}' for i in range(11, 21)]

all_tasks = []
skipped_no_text = 0
for spk in ENGLISH_SPEAKERS:
    spk_dir = f'{ESD_DIR}/{spk}'
    if not os.path.isdir(spk_dir):
        print(f'  warning: {spk_dir} not found, skipping')
        continue
    # Per-speaker transcripts file: "{spk}.txt" — tab-separated utt_id\ttext\temotion
    txt_files = glob.glob(f'{spk_dir}/*.txt')
    transcripts = {}
    for tf in txt_files:
        with open(tf, encoding='utf-8', errors='replace') as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) >= 2:
                    transcripts[parts[0]] = parts[1]  # parts[2] is emotion, already in path
    # Walk emotion subfolders (skip hidden, skip __MACOSX if any leaked)
    for emotion_dir in sorted(glob.glob(f'{spk_dir}/*')):
        if not os.path.isdir(emotion_dir):
            continue
        emotion = os.path.basename(emotion_dir)
        if emotion.startswith('.') or emotion.startswith('_'):
            continue  # skip junk
        for wav in sorted(glob.glob(f'{emotion_dir}/*.wav')):
            utt_id = os.path.splitext(os.path.basename(wav))[0]
            text = transcripts.get(utt_id, '')
            if not text:
                skipped_no_text += 1
            all_tasks.append((spk, emotion, wav, text, utt_id))

print(f'\nTotal English tasks: {len(all_tasks)}')
if skipped_no_text:
    print(f'  ({skipped_no_text} utterances had no transcript — still encoded; empty text OK for our delta computation)')
if all_tasks:
    spk, emo, _, text, utt_id = all_tasks[0]
    print(f'  sample: speaker={spk}  emotion={emo}  utt_id={utt_id}  text="{text[:60]}"')
    from collections import Counter
    emo_counts = Counter(t[1] for t in all_tasks)
    print(f'  emotions: {dict(emo_counts)}')  # expect ~3500 per emotion (10 speakers × 350)


In [ ]:
# Step 3: Load SPARC on GPU + speed check
from sparc import load_model
from pathlib import Path
import time

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Loading SPARC on {device}...')
coder = load_model('en', device=device)
print('SPARC loaded.')

# Pick 4 tasks not yet done for realistic timing
def make_stem(spk, emotion, utt_id):
    return f'{spk}_{emotion}_{utt_id}'

pending = [t for t in all_tasks if make_stem(t[0], t[1], t[4]) not in done_stems]
sample = pending[:4] if len(pending) >= 4 else all_tasks[:4]

if sample:
    _ = coder.encode(sample[0][2])  # warm-up
    t0 = time.time()
    for s in sample[1:4]:
        _ = coder.encode(s[2])
    avg = (time.time() - t0) / max(1, len(sample) - 1)
    remaining = len(pending)
    print(f'Per-file encode time: {avg:.2f}s')
    print(f'Remaining to encode: {remaining} files')
    print(f'Estimated total time: {avg * remaining / 60:.1f} minutes ({avg * remaining / 3600:.2f} hours)')
else:
    print('All files already encoded — nothing to do.')

In [ ]:
# Step 4: Encode + batched atomic sync to Drive with exponential backoff on failure.
#
# Sync every 100 files (SYNC_EVERY). Each file is uploaded atomically via tmp+rename
# and size-validated before the rename is committed. Failed syncs stay in the batch
# for retry on next flush.
#
# Per-file data stored: ema, pitch, loudness, spk_emb, emotion (string), text (string)
# File naming:  {speaker}_{emotion}_{utt_id}.npz  e.g. 0011_Happy_0011000351.npz

import numpy as np
import shutil
from pathlib import Path
from tqdm.notebook import tqdm

LOCAL_STAGE = '/content/staging'
os.makedirs(LOCAL_STAGE, exist_ok=True)

SYNC_EVERY   = 100
MIN_NPZ_SIZE = 500
MAX_FLUSH_ATTEMPTS = 5

success = 0
already_done = len(done_stems)
failed = 0
sync_failures = 0
sync_batch = []
t_start = time.time()


def sync_one(name, max_retries=4):
    src = Path(LOCAL_STAGE) / name
    dst = Path(DRIVE_FEATURES) / name
    tmp = Path(DRIVE_FEATURES) / (name + '.tmp')
    backoff = 5
    last_err = None
    for attempt in range(max_retries):
        try:
            shutil.copy2(str(src), str(tmp))
            size = tmp.stat().st_size
            if size < MIN_NPZ_SIZE:
                try: tmp.unlink()
                except Exception: pass
                raise RuntimeError(f'uploaded size {size} bytes below threshold {MIN_NPZ_SIZE}')
            os.replace(str(tmp), str(dst))
            src.unlink()
            return True
        except Exception as e:
            last_err = str(e)
            try:
                if tmp.exists(): tmp.unlink()
            except Exception:
                pass
            time.sleep(backoff)
            backoff = min(backoff * 2, 60)
    return False, last_err


def flush_sync_batch():
    global sync_failures
    synced_count = 0
    for name in list(sync_batch):
        result = sync_one(name)
        if result is True:
            sync_batch.remove(name)
            synced_count += 1
        else:
            _, err = result
            sync_failures += 1
            if sync_failures < 10:
                print(f'  sync fail {name}: {err}')
    return synced_count


pbar = tqdm(all_tasks, desc='Encoding', initial=already_done, total=len(all_tasks))

for (spk, emotion, wav_path, text, utt_id) in pbar:
    stem = make_stem(spk, emotion, utt_id)
    if stem in done_stems:
        continue

    stage_path = Path(LOCAL_STAGE) / f'{stem}.npz'

    try:
        code = coder.encode(wav_path)
        ema      = np.asarray(code['ema'],      dtype=np.float32)
        pitch    = np.asarray(code['pitch'],    dtype=np.float32).squeeze(-1)
        loudness = np.asarray(code['loudness'], dtype=np.float32).squeeze(-1)
        spk_emb  = np.asarray(code['spk_emb'],  dtype=np.float32)
        m = min(ema.shape[0], pitch.shape[0], loudness.shape[0])
        ema, pitch, loudness = ema[:m], pitch[:m], loudness[:m]
        # Save directly (local staging is ephemeral; no per-file atomicity needed
        # locally — atomicity on Drive side only)
        np.savez_compressed(
            str(stage_path),
            ema=ema, pitch=pitch, loudness=loudness, spk_emb=spk_emb,
            emotion=emotion, text=text, speaker=spk, utt_id=utt_id,
        )
        sync_batch.append(f'{stem}.npz')
        done_stems.add(stem)
        success += 1
    except Exception as e:
        if failed < 10:
            print(f'encode fail {stem}: {e}')
        failed += 1
        try:
            if stage_path.exists(): stage_path.unlink()
        except Exception:
            pass
        continue

    if len(sync_batch) >= SYNC_EVERY:
        flush_sync_batch()
        time.sleep(1)
        elapsed = time.time() - t_start
        rate = success / max(elapsed, 1)
        remaining = len(all_tasks) - success - already_done - failed
        eta_h = remaining / max(rate, 0.01) / 3600
        pbar.set_postfix(new=success, failed=failed, eta_h=f'{eta_h:.2f}')

# Final flush with bounded retries
for attempt in range(MAX_FLUSH_ATTEMPTS):
    if not sync_batch:
        break
    print(f'Final flush attempt {attempt+1}/{MAX_FLUSH_ATTEMPTS}: {len(sync_batch)} files to Drive...')
    synced = flush_sync_batch()
    print(f'  synced {synced}, remaining stuck: {len(sync_batch)}')
    if synced == 0 and sync_batch:
        time.sleep(10)

elapsed = time.time() - t_start
print(f'\nDone in {elapsed/3600:.2f}h ({elapsed/60:.1f} min)')
print(f'  encoded this session: {success}')
print(f'  already on Drive at start: {already_done}')
print(f'  encode failures: {failed}')
print(f'  sync attempts that errored: {sync_failures}')
print(f'  Drive now holds: {len(done_stems)} / {len(all_tasks)} files')

if sync_batch:
    print(f'\n⚠️  WARNING: {len(sync_batch)} files stuck in {LOCAL_STAGE}, NOT synced to Drive:')
    for n in sync_batch[:20]:
        print(f'    {n}')
    if len(sync_batch) > 20:
        print(f'    ... and {len(sync_batch) - 20} more')
    print('  Re-run this cell to retry upload. If runtime dies first, they will be re-encoded.')

## After encoding completes

### 1. Download from Drive

drive.google.com → `articulatory_tts/esd_features/` → right-click → Download (auto-zips).

### 2. Extract on your Mac

```bash
cd ~/projects/articulatory-tts/data
mkdir -p esd_features
cd esd_features
unzip ~/Downloads/esd_features-*.zip
find . -mindepth 2 -name '*.npz' -exec mv {} . \;
find . -type d -empty -delete
```

### 3. Run delta computation

```bash
python scripts/emotion/build_delta_table.py --mode esd \
    --esd-features-dir data/esd_features \
    --out-path data/emotion_deltas.npz
```

(The `esd` mode isn't implemented yet in `build_delta_table.py` — we'll write it once we have the features in hand.)